# Xem accuracy log — hỗ trợ CẢ 2 format

Tự nhận dạng format theo dòng đầu tiên:

1. **Train log** (`accuracy_log.jsonl` từ `QA_model/train.py`): mỗi dòng = 1 epoch — `accuracy`, `samples` (tất cả, gọn), `detail_samples` (~50 mẫu full). Tự chọn epoch accuracy CAO NHẤT.
2. **Ensemble log** (`ensemble_eval_log.jsonl` / `test_result.txt` từ `Ensemble_Model/inference.py`): mỗi dòng = 1 sample — `idx`, `correct`, `input{query, premises, options, premises_fol_gold}`, `fol_generated`, `gold`, `prediction{answer, premises_used, reasoning_steps, explanation}`, `latency`. Detail có cho MỌI sample.

Đổi `LOG_PATH` nếu cần; để trống/sai thì tự tìm file `*accuracy_log*.jsonl` hoặc `ensemble_eval_log*.jsonl` mới nhất dưới `outputs/`.

In [1]:
import json, glob, os

# ==== Đường dẫn log (sửa ở đây nếu cần) ====
LOG_PATH = r"E:\exact_2026\Exact_2026_Laplace-s_Red_Devils\Logic_Based_Educational_Queries_Project\outputs\Qwen_4B_Final_QA_result_test\ensemble_eval_log.jsonl"
if not os.path.isfile(LOG_PATH):
    cands = (glob.glob(os.path.join(os.getcwd(), '..', '**', '*accuracy_log*.jsonl'), recursive=True)
             + glob.glob(os.path.join(os.getcwd(), '..', '**', 'ensemble_eval_log*.jsonl'), recursive=True))
    if cands:
        LOG_PATH = max(cands, key=os.path.getmtime)
print('Đọc:', LOG_PATH)

rows = [json.loads(l) for l in open(LOG_PATH, encoding='utf-8') if l.strip()]

# ==== Tự nhận dạng format ====
#  A) train    : mỗi dòng = 1 epoch  {accuracy, samples, detail_samples}
#  B) ensemble : mỗi dòng = 1 sample {idx, correct, input, fol_generated, gold, prediction, latency}
FORMAT = 'train' if 'accuracy' in rows[0] else 'ensemble'
print(f'Format: {FORMAT}  ({len(rows)} dòng)')

if FORMAT == 'train':
    for i, r in enumerate(rows):
        print(f'  line {i}: accuracy={r["accuracy"]:.4f}  ({r["correct"]}/{r["total"]})')
    best_i = max(range(len(rows)), key=lambda i: rows[i]['accuracy'])
    best = rows[best_i]
    samples_brief = [{'idx': s['idx'], 'pred': s['pred_answer'], 'gold': s['gold_answer'], 'correct': s['correct']}
                     for s in best['samples']]
    details = best.get('detail_samples', [])
    header = (f'BEST epoch (line {best_i})  accuracy={best["accuracy"]:.4f}  '
              f'correct={best["correct"]}/{best["total"]}  avg_latency={best.get("avg_latency_sec", 0):.2f}s')
else:
    n = len(rows)
    correct = sum(r['correct'] for r in rows)
    avg_lat = {k: sum(r.get('latency', {}).get(k, 0) for r in rows) / n
               for k in ('fol_sec', 'qa_sec', 'total_sec')}
    samples_brief = [{'idx': r['idx'], 'pred': r['prediction']['answer'], 'gold': r['gold']['answer'], 'correct': r['correct']}
                     for r in rows]
    details = rows   # ensemble log có full detail cho MỌI sample
    header = (f'ENSEMBLE  accuracy={correct/n:.4f}  correct={correct}/{n}  '
              f'avg latency: FOL {avg_lat["fol_sec"]:.1f}s + QA {avg_lat["qa_sec"]:.1f}s = {avg_lat["total_sec"]:.1f}s/sample')

print('\n>>>', header)

Đọc: E:\exact_2026\Exact_2026_Laplace-s_Red_Devils\Logic_Based_Educational_Queries_Project\outputs\Qwen_4B_Final_QA_result_test\ensemble_eval_log.jsonl
Format: ensemble  (78 dòng)

>>> ENSEMBLE  accuracy=0.9103  correct=71/78  avg latency: FOL 5.1s + QA 6.2s = 11.2s/sample


In [2]:
# ==== TẤT CẢ sample (gọn: idx / pred / gold / đúng-sai) ====
TRUNC = 45  # cắt answer dài cho dễ nhìn (MCQ-text / công thức FOL)

print('=' * 100)
print(header)
print('=' * 100)
print(f'\n--- TẤT CẢ {len(samples_brief)} sample ---')
for s in samples_brief:
    mark = '✅' if s['correct'] else '❌'
    pred = str(s['pred'])[:TRUNC]
    gold = str(s['gold'])[:TRUNC]
    print(f'  {mark} idx={s["idx"]:<4} pred={pred:<{TRUNC}} | gold={gold}')

wrong_brief = [s for s in samples_brief if not s['correct']]
print(f'\n  SAI: {len(wrong_brief)}/{len(samples_brief)} mẫu — idx: {[s["idx"] for s in wrong_brief]}')

ENSEMBLE  accuracy=0.9103  correct=71/78  avg latency: FOL 5.1s + QA 6.2s = 11.2s/sample

--- TẤT CẢ 78 sample ---
  ✅ idx=0    pred=Yes                                           | gold=Yes
  ❌ idx=1    pred=No                                            | gold=Uncertain
  ✅ idx=2    pred=Yes                                           | gold=Yes
  ✅ idx=3    pred=Yes                                           | gold=Yes
  ✅ idx=4    pred=Yes                                           | gold=Yes
  ✅ idx=5    pred=Yes                                           | gold=Yes
  ✅ idx=6    pred=Yes                                           | gold=Yes
  ✅ idx=7    pred=Yes                                           | gold=Yes
  ✅ idx=8    pred=Yes                                           | gold=Yes
  ✅ idx=9    pred=Yes                                           | gold=Yes
  ✅ idx=10   pred=Some teaching methods are effective.          | gold=Some teaching methods are effective.
  ✅ idx=11   pred=Yes

In [3]:
# ==== FULL DETAIL: input + FOL gold/generated + GOLD vs PREDICTION ====
# In TẤT CẢ mẫu SAI trước + N_CORRECT_TO_SHOW mẫu đúng đầu tiên (None = in hết).
# Train log: detail chỉ có cho ~50 mẫu random/epoch. Ensemble log: đủ mọi sample.
N_CORRECT_TO_SHOW = 5

def show_detail(d):
    inp = d.get('input', {})
    print('=' * 92)
    head = f'idx={d["idx"]}    {"✅ CORRECT" if d["correct"] else "❌ WRONG"}'
    d_lat = d.get('latency')
    if d_lat:
        head += f'    (FOL {d_lat["fol_sec"]:.1f}s + QA {d_lat["qa_sec"]:.1f}s = {d_lat["total_sec"]:.1f}s)'
    print(head)
    print('=' * 92)

    q = inp.get('query') or inp.get('question') or ''
    print('QUESTION:\n  ' + str(q).replace('\n', '\n  '))

    prem = inp.get('premises') or inp.get('premises_nl') or []
    if prem:
        print('\nPREMISES (NL):')
        for j, t in enumerate(prem):
            print(f'  [{j}] {t}')

    opts = inp.get('options') or []
    if opts:
        print('\nOPTIONS:')
        for j, o in enumerate(opts):
            print(f'  {chr(65 + j)}. {o}')

    fol_gold = inp.get('premises_fol_gold') or inp.get('premises_fol') or []
    fol_gen = d.get('fol_generated')
    if fol_gold:
        print('\nPREMISES (FOL gold):')
        for j, f in enumerate(fol_gold):
            print(f'  [{j}] {f}')
    if fol_gen is not None:
        flag = '' if len(fol_gen) == len(fol_gold) else f'   ⚠️ LỆCH SỐ DÒNG: {len(fol_gen)} vs {len(fol_gold)} gold → premises_used có thể lệch index!'
        print(f'\nFOL GENERATED (model 1):{flag}')
        for j, f in enumerate(fol_gen):
            print(f'  [{j}] {f}')

    for who in ('gold', 'prediction'):
        b = d.get(who, {})
        print(f'\n── {who.upper()} ──')
        print(f'  answer         : {b.get("answer")}')
        if b.get('premises_used') is not None:
            print(f'  premises_used  : {b.get("premises_used")}')
        steps = b.get('reasoning_steps') or []
        if steps:
            print('  reasoning_steps:')
            for s in steps:
                print(f'      - {s}')
        print(f'  explanation    : {b.get("explanation", "")}')
    print()

wrong_d = [d for d in details if not d['correct']]
right_d = [d for d in details if d['correct']]
shown = wrong_d + (right_d if N_CORRECT_TO_SHOW is None else right_d[:N_CORRECT_TO_SHOW])
print(f'{"#" * 92}\n#  FULL DETAIL — {len(wrong_d)} mẫu SAI + {len(shown) - len(wrong_d)} mẫu đúng (chỉnh N_CORRECT_TO_SHOW)\n{"#" * 92}\n')
for d in shown:
    show_detail(d)

############################################################################################
#  FULL DETAIL — 7 mẫu SAI + 5 mẫu đúng (chỉnh N_CORRECT_TO_SHOW)
############################################################################################

idx=1    ❌ WRONG    (FOL 3.5s + QA 6.2s = 9.6s)
QUESTION:
  Do all students take a qualifying exam?

PREMISES (NL):
  [0] There exists at least one student who has completed a research project.
  [1] There exists at least one student who has taken a qualifying exam.
  [2] If a student has completed a research project, then they have submitted a thesis.
  [3] If at least one student has taken a qualifying exam, then at least one student has completed a research project.
  [4] If completing a research project ensures submitting a thesis, then the existence of a student taking a qualifying exam guarantees that at least one student has completed a research project.
  [5] If a student has not completed a research project, then they have not t